In [11]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [12]:
import pandas as pd
import pickle
import tensorflow as tf

# Load dataset
data = pd.read_csv("../Data/Churn_Modelling.csv")

# Load model
model = tf.keras.models.load_model("../Data/model.h5")

# Load encoders & scaler
with open("../Data/onehot_encoder_geo.pkl", "rb") as f:
    onehot_encoder_geo = pickle.load(f)

with open("../Data/label_encoder_gender.pkl", "rb") as f:
    label_encoder_gender = pickle.load(f)

with open("../Data/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

c:\Users\ADMIN\miniconda3\envs\ann_env\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\ADMIN\miniconda3\envs\ann_env\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\ADMIN\miniconda3\envs\ann_env\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid result

In [13]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [14]:
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [15]:
## Encode categorical variables
input_df['Gender']=label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [16]:

# Encode the 'Geography' feature using the fitted one-hot encoder and convert it to one-hot array
geo_encoded = onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()

# Convert the one-hot encoded array into a pandas DataFrame
# Columns are named according to the categories in 'Geography'
geo_encoded_df = pd.DataFrame(
    geo_encoded, 
    columns=onehot_encoder_geo.get_feature_names_out(['Geography'])
)
geo_encoded_df


c:\Users\ADMIN\miniconda3\envs\ann_env\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [17]:
## concatination one hot encoded 
input_df=pd.concat([input_df.drop("Geography",axis=1),geo_encoded_df],axis=1)
input_df





,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [18]:
scaler.feature_names_in_


array(['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance',
       'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'Exited',
       'Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [19]:
## Scaling the input data
input_scaled=scaler.transform(input_df)
input_scaled

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- EstimatedSalary
Feature names seen at fit time, yet now missing:
- Exited


In [ ]:
## PRedict churn
prediction=model.predict(input_scaled)
prediction
# under there is written why we got this answer 0.03066216

1/1 [==============================] - 0s 95ms/step


array([[0.09015328]], dtype=float32)

In [ ]:
prediction_proba = prediction[0][0]

In [ ]:
prediction_proba

0.090153284

In [ ]:
if prediction_proba > 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.
